In [ ]:
! pip install seaborn scikit-learn


In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk(r'C:\Users\SCLuser\Desktop\python-excercise\notebooks'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

C:\Users\SCLuser\Desktop\python-excercise\notebooks\data_overview.png
C:\Users\SCLuser\Desktop\python-excercise\notebooks\notebooka2259bb61f.ipynb
C:\Users\SCLuser\Desktop\python-excercise\notebooks\time_series_covid19_confirmed_global.csv
C:\Users\SCLuser\Desktop\python-excercise\notebooks\time_series_covid19_deaths_global.csv
C:\Users\SCLuser\Desktop\python-excercise\notebooks\time_series_covid19_recovered_global.csv


In [8]:
# COVID-19 Global Pandemic Analysis: Comprehensive Data Science Project
# A multi-dimensional analysis of global COVID-19 patterns with advanced visualizations and predictive modeling

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.signal import find_peaks
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Suppress warnings
warnings.filterwarnings('ignore')

# Set style for matplotlib
plt.style.use('fivethirtyeight')
sns.set_style('whitegrid')

# Custom color palette
colors = ["#ff5a5f", "#00a699", "#3c3c3c", "#767676", "#484848"]
sns.set_palette(sns.color_palette(colors))

print("Loading COVID-19 datasets...")

# Load the datasets
confirmed_df = pd.read_csv('time_series_covid19_confirmed_global.csv')
deaths_df = pd.read_csv('time_series_covid19_deaths_global.csv')
recovered_df = pd.read_csv('time_series_covid19_recovered_global.csv')

# Display basic info about the datasets
print(f"Confirmed cases data shape: {confirmed_df.shape}")
print(f"Deaths data shape: {deaths_df.shape}")
print(f"Recovered cases data shape: {recovered_df.shape}")

# Check the first few rows of each dataset
print("\nFirst few rows of confirmed cases data:")
print(confirmed_df.head(3))

# Function to preprocess and transform the data
def preprocess_data(df, case_type):
    # Create a copy of the dataframe
    df_processed = df.copy()
    
    # Extract location columns
    location_cols = ['Province/State', 'Country/Region', 'Lat', 'Long']
    
    # Melt the dataframe to convert from wide to long format
    melted_df = pd.melt(
        df_processed, 
        id_vars=location_cols,
        var_name='Date', 
        value_name=case_type
    )
    
    # Convert date column to datetime
    melted_df['Date'] = pd.to_datetime(melted_df['Date'])
    
    return melted_df

# Process all datasets
confirmed_long = preprocess_data(confirmed_df, 'Confirmed')
deaths_long = preprocess_data(deaths_df, 'Deaths')
recovered_long = preprocess_data(recovered_df, 'Recovered')

# Merge the datasets
covid_df = confirmed_long.merge(
    deaths_long[['Province/State', 'Country/Region', 'Date', 'Deaths']], 
    on=['Province/State', 'Country/Region', 'Date'], 
    how='left'
)

covid_df = covid_df.merge(
    recovered_long[['Province/State', 'Country/Region', 'Date', 'Recovered']], 
    on=['Province/State', 'Country/Region', 'Date'], 
    how='left'
)

# Fill NaN values in Recovered column with 0
covid_df['Recovered'] = covid_df['Recovered'].fillna(0)

# Create additional metrics
covid_df['Active'] = covid_df['Confirmed'] - covid_df['Deaths'] - covid_df['Recovered']
covid_df['Mortality Rate (%)'] = (covid_df['Deaths'] / covid_df['Confirmed'] * 100).round(2)
covid_df['Recovery Rate (%)'] = (covid_df['Recovered'] / covid_df['Confirmed'] * 100).round(2)

# Handle NaN and Inf values in calculated rates
covid_df.replace([np.inf, -np.inf], np.nan, inplace=True)
covid_df['Mortality Rate (%)'] = covid_df['Mortality Rate (%)'].fillna(0)
covid_df['Recovery Rate (%)'] = covid_df['Recovery Rate (%)'].fillna(0)

print("\nProcessed dataset preview:")
print(covid_df.head())

# Verify data types to avoid summing non-numeric columns
print("\nData types in processed dataset:")
print(covid_df.dtypes)

# Aggregate data by country and date
country_data = covid_df.groupby(['Country/Region', 'Date'])[
    ['Confirmed', 'Deaths', 'Recovered', 'Active']
].sum().reset_index()

# Calculate daily changes
country_data = country_data.sort_values(['Country/Region', 'Date'])
country_data['Daily New Cases'] = country_data.groupby('Country/Region')['Confirmed'].diff().fillna(0)
country_data['Daily New Deaths'] = country_data.groupby('Country/Region')['Deaths'].diff().fillna(0)
country_data['Daily New Recovered'] = country_data.groupby('Country/Region')['Recovered'].diff().fillna(0)

# Replace negative values with 0 (data corrections in source)
for col in ['Daily New Cases', 'Daily New Deaths', 'Daily New Recovered']:
    country_data[col] = country_data[col].clip(lower=0)

# Calculate 7-day moving averages
country_data['7-Day Avg New Cases'] = country_data.groupby('Country/Region')['Daily New Cases'].rolling(7).mean().reset_index(0, drop=True)
country_data['7-Day Avg New Deaths'] = country_data.groupby('Country/Region')['Daily New Deaths'].rolling(7).mean().reset_index(0, drop=True)

print("\n---- Part 1: Global COVID-19 Overview ----")

# Get latest date in the dataset
latest_date = covid_df['Date'].max()
print(f"\nData available up to: {latest_date.strftime('%Y-%m-%d')}")

# Global totals as of the latest date - FIX: Only sum numeric columns
global_totals = covid_df[covid_df['Date'] == latest_date][['Confirmed', 'Deaths', 'Recovered', 'Active']].sum()
print("\nGlobal COVID-19 Totals:")
print(f"Total Confirmed Cases: {global_totals['Confirmed']:,.0f}")
print(f"Total Deaths: {global_totals['Deaths']:,.0f}")
print(f"Total Recovered: {global_totals['Recovered']:,.0f}")
print(f"Total Active Cases: {global_totals['Active']:,.0f}")
print(f"Global Mortality Rate: {(global_totals['Deaths'] / global_totals['Confirmed'] * 100):.2f}%")
print(f"Global Recovery Rate: {(global_totals['Recovered'] / global_totals['Confirmed'] * 100):.2f}%")

# Global trend over time
global_trend = covid_df.groupby('Date')[['Confirmed', 'Deaths', 'Recovered', 'Active']].sum().reset_index()
global_trend['Daily New Cases'] = global_trend['Confirmed'].diff().fillna(0)
global_trend['Daily New Deaths'] = global_trend['Deaths'].diff().fillna(0)
global_trend['Daily New Recovered'] = global_trend['Recovered'].diff().fillna(0)

# Replace negative values with 0
for col in ['Daily New Cases', 'Daily New Deaths', 'Daily New Recovered']:
    global_trend[col] = global_trend[col].clip(lower=0)

# Calculate 7-day moving averages
global_trend['7-Day Avg New Cases'] = global_trend['Daily New Cases'].rolling(7).mean()
global_trend['7-Day Avg New Deaths'] = global_trend['Daily New Deaths'].rolling(7).mean()
global_trend['7-Day Avg New Recovered'] = global_trend['Daily New Recovered'].rolling(7).mean()

# Plotting global cumulative counts
plt.figure(figsize=(16, 8))
plt.plot(global_trend['Date'], global_trend['Confirmed'], label='Confirmed', linewidth=2)
plt.plot(global_trend['Date'], global_trend['Deaths'], label='Deaths', linewidth=2)
plt.plot(global_trend['Date'], global_trend['Recovered'], label='Recovered', linewidth=2)
plt.plot(global_trend['Date'], global_trend['Active'], label='Active', linewidth=2)
plt.title('Global COVID-19 Cumulative Counts', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.yscale('log')
plt.tight_layout()
plt.savefig('global_cumulative_counts.png', dpi=300, bbox_inches='tight')
plt.close()

# Plotting global daily new cases and deaths with moving averages
fig, axes = plt.subplots(2, 1, figsize=(16, 12), sharex=True)

# New Cases plot
axes[0].bar(global_trend['Date'], global_trend['Daily New Cases'], 
            alpha=0.3, color='#3498db', label='Daily New Cases')
axes[0].plot(global_trend['Date'], global_trend['7-Day Avg New Cases'], 
             color='#2980b9', linewidth=2, label='7-Day Moving Average')
axes[0].set_title('Global Daily New COVID-19 Cases', fontsize=16)
axes[0].set_ylabel('Number of Cases', fontsize=12)
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# New Deaths plot
axes[1].bar(global_trend['Date'], global_trend['Daily New Deaths'], 
            alpha=0.3, color='#e74c3c', label='Daily New Deaths')
axes[1].plot(global_trend['Date'], global_trend['7-Day Avg New Deaths'], 
             color='#c0392b', linewidth=2, label='7-Day Moving Average')
axes[1].set_title('Global Daily New COVID-19 Deaths', fontsize=16)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Number of Deaths', fontsize=12)
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('global_daily_metrics.png', dpi=300, bbox_inches='tight')
plt.close()

print("\n---- Part 2: Country-level Analysis ----")

# Get latest data for each country
latest_country_data = country_data[country_data['Date'] == latest_date].copy()
latest_country_data = latest_country_data.sort_values('Confirmed', ascending=False)

# Display top 15 countries by confirmed cases
print("\nTop 15 Countries by Confirmed Cases:")
top_15_countries = latest_country_data.head(15)
print(top_15_countries[['Country/Region', 'Confirmed', 'Deaths', 'Recovered', 'Active']].reset_index(drop=True))

# Plot top 10 countries by confirmed cases
plt.figure(figsize=(14, 8))
top_10_confirmed = latest_country_data.head(10)
plt.barh(top_10_confirmed['Country/Region'], top_10_confirmed['Confirmed'])
plt.title('Top 10 Countries by Confirmed COVID-19 Cases', fontsize=16)
plt.xlabel('Number of Confirmed Cases', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.grid(True, alpha=0.3)
plt.gca().invert_yaxis()  # To have the highest value at the top
plt.tight_layout()
plt.savefig('top_10_countries_confirmed.png', dpi=300, bbox_inches='tight')
plt.close()

# Plot top 10 countries by death count
plt.figure(figsize=(14, 8))
top_10_deaths = latest_country_data.sort_values('Deaths', ascending=False).head(10)
plt.barh(top_10_deaths['Country/Region'], top_10_deaths['Deaths'], color='darkred')
plt.title('Top 10 Countries by COVID-19 Deaths', fontsize=16)
plt.xlabel('Number of Deaths', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.grid(True, alpha=0.3)
plt.gca().invert_yaxis()  # To have the highest value at the top
plt.tight_layout()
plt.savefig('top_10_countries_deaths.png', dpi=300, bbox_inches='tight')
plt.close()

print("\n---- Part 3: Advanced Analysis & Visualizations ----")

# Create a normalized dataset for cross-country comparison
# Select countries with significant case numbers for analysis
major_countries = ['US', 'India', 'Brazil', 'Russia', 'France', 
                  'United Kingdom', 'Italy', 'Spain', 'Germany', 
                  'Argentina', 'Mexico', 'South Africa', 'Turkey']

# Filter data for selected countries
major_countries_data = country_data[country_data['Country/Region'].isin(major_countries)].copy()

# Create a heatmap showing case growth patterns
# First, resample to monthly data to reduce complexity
pivot_data = pd.DataFrame()
for country in major_countries:
    country_slice = major_countries_data[major_countries_data['Country/Region'] == country]
    # Resample to monthly data
    monthly = country_slice.set_index('Date').resample('M')['Daily New Cases'].mean().reset_index()
    monthly['Country/Region'] = country
    pivot_data = pd.concat([pivot_data, monthly])

# Create pivot table
pivot_table = pivot_data.pivot_table(
    index='Country/Region',
    columns=pd.Grouper(key='Date', freq='M'),
    values='Daily New Cases'
)

# Plot heatmap
plt.figure(figsize=(20, 10))
sns.heatmap(pivot_table, cmap='YlOrRd', linewidths=0.5, linecolor='white')
plt.title('Average Daily New Cases by Month and Country', fontsize=16)
plt.xlabel('Month', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('cases_heatmap_by_country_month.png', dpi=300, bbox_inches='tight')
plt.close()

# Analyzing correlation between Mortality Rate and Recovery Rate
# Calculate rates at country level
latest_full_data = covid_df[covid_df['Date'] == latest_date].copy()
country_rates = latest_full_data.groupby('Country/Region').agg({
    'Confirmed': 'sum',
    'Deaths': 'sum',
    'Recovered': 'sum'
}).reset_index()

# Calculate rates
country_rates['Mortality Rate (%)'] = (country_rates['Deaths'] / country_rates['Confirmed'] * 100).round(2)
country_rates['Recovery Rate (%)'] = (country_rates['Recovered'] / country_rates['Confirmed'] * 100).round(2)
country_rates = country_rates[country_rates['Confirmed'] > 10000]  # Filter for countries with significant cases

# Handle NaN values
country_rates.replace([np.inf, -np.inf], np.nan, inplace=True)
country_rates = country_rates.dropna()

# Scatter plot of mortality vs recovery rates
plt.figure(figsize=(14, 8))
plt.scatter(
    country_rates['Mortality Rate (%)'], 
    country_rates['Recovery Rate (%)'],
    s=country_rates['Confirmed']/10000,  # Size based on case count
    alpha=0.6
)

# Add country labels to the largest points
for i, row in country_rates.sort_values('Confirmed', ascending=False).head(15).iterrows():
    plt.annotate(
        row['Country/Region'],
        xy=(row['Mortality Rate (%)'], row['Recovery Rate (%)']),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=9
    )

plt.title('Relationship between Mortality Rate and Recovery Rate by Country', fontsize=16)
plt.xlabel('Mortality Rate (%)', fontsize=12)
plt.ylabel('Recovery Rate (%)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('mortality_vs_recovery_scatter.png', dpi=300, bbox_inches='tight')
plt.close()

# Time series analysis for the top 5 most affected countries
top_5_countries = latest_country_data['Country/Region'].head(5).tolist()
top_5_data = country_data[country_data['Country/Region'].isin(top_5_countries)].copy()

# Plotting cases over time for top 5 countries
plt.figure(figsize=(16, 10))

for country in top_5_countries:
    country_slice = top_5_data[top_5_data['Country/Region'] == country]
    plt.plot(country_slice['Date'], country_slice['7-Day Avg New Cases'], label=country, linewidth=2)

plt.title('7-Day Average of New Cases for Top 5 Countries', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('7-Day Average of New Cases', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig('top_5_countries_cases_time_series.png', dpi=300, bbox_inches='tight')
plt.close()

print("\n---- Part 4: Correlation Analysis ----")

# Create correlation matrix of numeric features
corr_data = covid_df[covid_df['Date'] == latest_date][['Confirmed', 'Deaths', 'Recovered', 'Active', 'Mortality Rate (%)', 'Recovery Rate (%)']].copy()
corr_matrix = corr_data.corr()

# Plot correlation matrix
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of COVID-19 Metrics', fontsize=16)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.close()

print("\n---- Part 5: Trend Analysis ----")

# Calculate daily changes in active cases
global_trend['Daily Active Change'] = global_trend['Active'].diff()

# Plot active cases trend
plt.figure(figsize=(16, 8))
plt.plot(global_trend['Date'], global_trend['Active'], label='Global Active Cases', color='orange', linewidth=2)
plt.title('Global COVID-19 Active Cases Trend', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Number of Active Cases', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig('global_active_cases_trend.png', dpi=300, bbox_inches='tight')
plt.close()

# Create weekly growth rates
global_trend['Weekly Growth Rate (%)'] = (global_trend['Confirmed'].pct_change(7) * 100).round(2)

# Plot weekly growth rate
plt.figure(figsize=(16, 8))
plt.plot(global_trend['Date'], global_trend['Weekly Growth Rate (%)'], color='purple', linewidth=2)
plt.axhline(y=0, color='red', linestyle='--', alpha=0.7)
plt.title('Global COVID-19 Weekly Growth Rate', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Weekly Growth Rate (%)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('global_weekly_growth_rate.png', dpi=300, bbox_inches='tight')
plt.close()

print("\n---- Part 6: Regional Analysis ----")

# Define regions for analysis
regions = {
    'North America': ['US', 'Canada', 'Mexico'],
    'South America': ['Brazil', 'Argentina', 'Colombia', 'Peru', 'Chile'],
    'Europe': ['United Kingdom', 'France', 'Germany', 'Italy', 'Spain', 'Russia'],
    'Asia': ['India', 'China', 'Japan', 'South Korea', 'Iran', 'Turkey'],
    'Africa': ['South Africa', 'Egypt', 'Morocco', 'Nigeria', 'Ethiopia'],
    'Oceania': ['Australia', 'New Zealand']
}

# Create a regional dataset
regional_data = pd.DataFrame()
for region, countries in regions.items():
    region_slice = country_data[country_data['Country/Region'].isin(countries)]
    # Group by date and calculate sums
    region_by_date = region_slice.groupby('Date')[['Confirmed', 'Deaths', 'Recovered', 'Active']].sum().reset_index()
    region_by_date['Region'] = region
    regional_data = pd.concat([regional_data, region_by_date])

# Calculate daily changes
regional_data = regional_data.sort_values(['Region', 'Date'])
regional_data['Daily New Cases'] = regional_data.groupby('Region')['Confirmed'].diff().fillna(0)
regional_data['Daily New Deaths'] = regional_data.groupby('Region')['Deaths'].diff().fillna(0)

# Replace negative values
for col in ['Daily New Cases', 'Daily New Deaths']:
    regional_data[col] = regional_data[col].clip(lower=0)

# Calculate 7-day moving averages
regional_data['7-Day Avg New Cases'] = regional_data.groupby('Region')['Daily New Cases'].rolling(7).mean().reset_index(0, drop=True)

# Plot regional comparison
plt.figure(figsize=(16, 10))

for region in regional_data['Region'].unique():
    region_slice = regional_data[regional_data['Region'] == region]
    plt.plot(region_slice['Date'], region_slice['7-Day Avg New Cases'], label=region, linewidth=2)

plt.title('COVID-19 Cases by Region (7-Day Moving Average)', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('7-Day Average of New Cases', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig('regional_cases_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

print("\n---- Part 7: Cluster Analysis ----")

# Prepare data for clustering
# Get data for the most recent date
cluster_data = latest_country_data[['Country/Region', 'Confirmed', 'Deaths', 'Recovered', 'Active']].copy()
cluster_data = cluster_data[cluster_data['Confirmed'] > 10000]  # Filter countries with significant cases

# Calculate additional metrics for clustering
cluster_data['Deaths_per_Case'] = cluster_data['Deaths'] / cluster_data['Confirmed']
cluster_data['Recovery_Rate'] = cluster_data['Recovered'] / cluster_data['Confirmed']
cluster_data['Active_Rate'] = cluster_data['Active'] / cluster_data['Confirmed']

# Handle NaN values
cluster_data.replace([np.inf, -np.inf], np.nan, inplace=True)
cluster_data = cluster_data.dropna()

# Select features for clustering
features = ['Deaths_per_Case', 'Recovery_Rate', 'Active_Rate']
X = cluster_data[features].copy()

# Scale the data
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Determine optimal number of clusters using the elbow method
inertia = []
k_range = range(1, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

# Plot the elbow curve
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia, 'bo-')
plt.title('Elbow Method for Optimal k', fontsize=16)
plt.xlabel('Number of Clusters (k)', fontsize=12)
plt.ylabel('Inertia', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('elbow_method.png', dpi=300, bbox_inches='tight')
plt.close()

# Choose k=4 based on the elbow curve
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
cluster_data['Cluster'] = kmeans.fit_predict(X_scaled)

# Apply PCA for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
cluster_data['PCA1'] = X_pca[:, 0]
cluster_data['PCA2'] = X_pca[:, 1]

# Plot the clusters
plt.figure(figsize=(14, 10))
for cluster_id in range(k):
    cluster_points = cluster_data[cluster_data['Cluster'] == cluster_id]
    plt.scatter(
        cluster_points['PCA1'], 
        cluster_points['PCA2'],
        s=cluster_points['Confirmed'] / 10000,
        label=f'Cluster {cluster_id}',
        alpha=0.6
    )

# Annotate the plot with country names for the largest countries
for i, row in cluster_data.sort_values('Confirmed', ascending=False).head(20).iterrows():
    plt.annotate(
        row['Country/Region'],
        xy=(row['PCA1'], row['PCA2']),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=9
    )

plt.title('Country Clusters based on COVID-19 Metrics', fontsize=16)
plt.xlabel('Principal Component 1', fontsize=12)
plt.ylabel('Principal Component 2', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig('country_clusters.png', dpi=300, bbox_inches='tight')
plt.close()

# Analyze the clusters
cluster_analysis = cluster_data.groupby('Cluster')[features + ['Confirmed', 'Deaths', 'Recovered', 'Active']].mean()
print("\nCluster Analysis (Average Values):")
print(cluster_analysis)

# Identify countries in each cluster
for i in range(k):
    countries = cluster_data[cluster_data['Cluster'] == i]['Country/Region'].tolist()
    print(f"\nCluster {i} Countries ({len(countries)} countries):")
    print(", ".join(countries[:10]) + ("..." if len(countries) > 10 else ""))

print("\n---- Part 8: Wave Analysis ----")

# Analyze global waves of the pandemic
wave_data = global_trend[['Date', '7-Day Avg New Cases']].copy()

# Use peak detection algorithm
peaks, _ = find_peaks(wave_data['7-Day Avg New Cases'], height=wave_data['7-Day Avg New Cases'].max()*0.2, distance=30)

# Plot waves
plt.figure(figsize=(16, 8))
plt.plot(wave_data['Date'], wave_data['7-Day Avg New Cases'], label='7-Day Avg New Cases', color='blue')

# Mark peaks
for i, peak in enumerate(peaks):
    plt.axvline(x=wave_data['Date'].iloc[peak], color='red', linestyle='--', alpha=0.5)
    plt.text(wave_data['Date'].iloc[peak], wave_data['7-Day Avg New Cases'].max()*0.95, 
            f'Wave {i+1}', 
            rotation=90, verticalalignment='top')

plt.title('COVID-19 Global Waves Analysis', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('7-Day Average of New Cases', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('global_waves_analysis.png', dpi=300, bbox_inches='tight')
plt.close()

print(f"Detected {len(peaks)} major global COVID-19 waves")

print("\n---- Part 9: Time Series Forecasting ----")

# We'll use a simple time series forecasting approach without requiring Prophet
# Prepare time series data for forecasting
forecast_data = global_trend[['Date', '7-Day Avg New Cases']].copy()
forecast_data.set_index('Date', inplace=True)

# Use a simple moving average model for forecasting
window_size = 14  # Use a 14-day window
forecast_data['MA_14'] = forecast_data['7-Day Avg New Cases'].rolling(window=window_size).mean()

# Fill NaN values in the beginning
forecast_data['MA_14'] = forecast_data['MA_14'].fillna(forecast_data['7-Day Avg New Cases'])

# Generate a forecast for the next 30 days
last_date = forecast_data.index[-1]
forecast_dates = pd.date_range(start=last_date + timedelta(days=1), periods=30)
forecast_values = [forecast_data['MA_14'].iloc[-window_size:].mean()] * 30

# Create a DataFrame for the forecast
future_forecast = pd.DataFrame({
    'Date': forecast_dates,
    'Forecast': forecast_values
})

# Plot the forecast
plt.figure(figsize=(16, 8))
plt.plot(forecast_data.index, forecast_data['7-Day Avg New Cases'], label='Historical Data', color='blue')
plt.plot(future_forecast['Date'], future_forecast['Forecast'], label='Forecast', color='red', linestyle='--')
plt.title('COVID-19 Global Cases Forecast (Simple Moving Average)', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('7-Day Average of New Cases', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig('global_forecast.png', dpi=300, bbox_inches='tight')
plt.close()

print("\n---- Part 10: Death Rate Analysis ----")

# Calculate CFR (Case Fatality Rate) over time
global_trend['CFR (%)'] = (global_trend['Deaths'] / global_trend['Confirmed'] * 100).round(2)

# Plot CFR over time
plt.figure(figsize=(16, 8))
plt.plot(global_trend['Date'], global_trend['CFR (%)'], color='darkred', linewidth=2)
plt.title('Global COVID-19 Case Fatality Rate Over Time', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Case Fatality Rate (%)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('global_cfr_trend.png', dpi=300, bbox_inches='tight')
plt.close()

print("\n---- Part 11: Conclusion and Summary ----")

print("\nKey Findings from COVID")

Loading COVID-19 datasets...
Confirmed cases data shape: (289, 1147)
Deaths data shape: (289, 1147)
Recovered cases data shape: (274, 1147)

First few rows of confirmed cases data:
  Province/State Country/Region       Lat       Long  1/22/20  1/23/20  \
0            NaN    Afghanistan  33.93911  67.709953        0        0   
1            NaN        Albania  41.15330  20.168300        0        0   
2            NaN        Algeria  28.03390   1.659600        0        0   

   1/24/20  1/25/20  1/26/20  1/27/20  ...  2/28/23  3/1/23  3/2/23  3/3/23  \
0        0        0        0        0  ...   209322  209340  209358  209362   
1        0        0        0        0  ...   334391  334408  334408  334427   
2        0        0        0        0  ...   271441  271448  271463  271469   

   3/4/23  3/5/23  3/6/23  3/7/23  3/8/23  3/9/23  
0  209369  209390  209406  209436  209451  209451  
1  334427  334427  334427  334427  334443  334457  
2  271469  271477  271477  271490  271494  271496